In [ ]:
%%capture
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu124
!pip install unsloth
!pip install --upgrade transformers==4.56.1 "huggingface_hub>=0.34.0" "datasets>=3.4.1,<4.0.0"

In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from huggingface_hub import login
from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from unsloth.chat_templates import train_on_responses_only
from transformers import AutoTokenizer, StoppingCriteria, StoppingCriteriaList

login('your_huggingface_auth_token_here')

In [ ]:
model, _ = FastLanguageModel.from_pretrained(
    model_name="./models/ViLegalQwen2.5-1.5B-Base", # or ViLegalQwen3-1.7B-Base with Qwen/Qwen3-1.7B-Base's tokenizer
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=True,
    token = "your_huggingface_auth_token_here",
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    modules_to_save=['embed_tokens', 'lm_head']
)

In [ ]:
df_train = pd.read_csv("./datasets/ViLegalTF/ViLegalTF_train.csv")
df_val = pd.read_csv("./datasets/ViLegalTF/ViLegalTF_val.csv")
df_test = pd.read_csv("./datasets/ALQAC/ALQAC-TF.csv")

df_train = df_train[["context", "question", "answer"]]
df_val = df_val[["context", "question", "answer"]]

df_train["answer"] = df_train["answer"].apply(lambda x: "A" if x == "Đúng" else "B")
df_val["answer"] = df_val["answer"].apply(lambda x: "A" if x == "Đúng" else "B")

df_test["answer"] = df_test["answer"].apply(lambda x: "A" if x == "Đúng" else "B")

In [ ]:
len(df_train), len(df_val), len(df_test)

In [ ]:
df_train["answer"].value_counts()

In [ ]:
df_val["answer"].value_counts()

In [ ]:
df_test["answer"].value_counts()

In [ ]:
SYSTEM_PROMPT = "Bạn là chuyên gia pháp luật Việt Nam có nhiều kinh nghiệm trong việc đánh giá tính chính xác của các phát biểu pháp lý dựa trên văn bản pháp luật được cung cấp."
USER_PROMPT = "Văn bản pháp luật:\n{context}\n\nPhát biểu:\n{question}\n\nCác lựa chọn:\nA. Đúng\nB. Sai\n\nDựa vào văn bản pháp luật trên, hãy xác định phát biểu là đúng hay sai."

In [ ]:
def generate_train_instruction(system_prompt, user_prompt, context, question, answer):
    instruction = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt.format(context=context, question=question)}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"
    return instruction

def generate_test_instruction(system_prompt, user_prompt, context, question):
    instruction = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt.format(context=context, question=question)}<|im_end|>\n<|im_start|>assistant\n"
    return instruction

In [ ]:
for index, values in tqdm(df_train.iterrows(), total=len(df_train), desc="Generating intruction prompt for training..."):
    CONTEXT = df_train['context'][index]
    QUESTION = df_train['question'][index]
    ANSWER = df_train['answer'][index]

    df_train.at[index, "instruction"] = generate_train_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        context=CONTEXT,
        question=QUESTION,
        answer=ANSWER
    )

for index, values in tqdm(df_val.iterrows(), total=len(df_val), desc="Generating intruction prompt for validation..."):
    CONTEXT = df_val['context'][index]
    QUESTION = df_val['question'][index]
    ANSWER = df_val['answer'][index]

    df_val.at[index, "instruction"] = generate_train_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        context=CONTEXT,
        question=QUESTION,
        answer=ANSWER
    )

for index, values in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating intruction prompt for testing..."):
    CONTEXT = df_test['context'][index]
    QUESTION = df_test['question'][index]

    df_test.at[index, "instruction"] = generate_test_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        context=CONTEXT,
        question=QUESTION
    )

In [ ]:
df_train["len_instruction"] = df_train["instruction"].apply(lambda x: len(x.split(" ")))
df_val["len_instruction"] = df_val["instruction"].apply(lambda x: len(x.split(" ")))
df_test["len_instruction"] = df_test["instruction"].apply(lambda x: len(x.split(" ")))

In [ ]:
df_train["answer"].value_counts()

In [ ]:
df_val["answer"].value_counts()

In [ ]:
df_test["answer"].value_counts()

In [ ]:
df_train["len_instruction"].max(), df_val["len_instruction"].max(), df_test["len_instruction"].max()

In [ ]:
dataset_train = Dataset.from_pandas(df_train, preserve_index=False)
dataset_val = Dataset.from_pandas(df_val, preserve_index=False)
dataset_test = Dataset.from_pandas(df_test, preserve_index=False)

In [ ]:
def convert_to_text_format(dataset):
    def map_func(examples):
        return {"text": examples["instruction"]}
    
    return dataset.map(map_func, batched=True, remove_columns=dataset.column_names)

dataset_train_formatted = convert_to_text_format(dataset_train)
dataset_val_formatted = convert_to_text_format(dataset_val)
dataset_test_formatted = convert_to_text_format(dataset_test)

# Simple formatting function
def formatting_func(examples):
    return {"text": examples["text"]}

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train_formatted,
    eval_dataset=dataset_val_formatted,
    dataset_text_field="text",
    max_seq_length=2048,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=True,
    formatting_func=formatting_func,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        learning_rate=3e-4,
        fp16=True,
        bf16=False,
        logging_steps=100,
        eval_strategy="steps",
        eval_steps=100, 
        optim="paged_adamw_32bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine_warmup_with_min_lr",
        warmup_ratio = 0.1,
        lr_scheduler_kwargs={"min_lr": 1e-5},
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_strategy="no",
    ),
)

In [ ]:
trainer_stats = trainer.train()

# Inference

In [ ]:
class EndOfConversationCriteria(StoppingCriteria):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.end_token_id = tokenizer.encode("<|im_end|>", add_special_tokens=False)[0]
    
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1] == self.end_token_id

FastLanguageModel.for_inference(model)

stopping_criteria = StoppingCriteriaList([EndOfConversationCriteria(tokenizer)])

In [ ]:
df_test["generated_answer"] = ""

for index, values in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating answer for test set..."):
    instruction = values["instruction"]
    gold_answer = values["answer"]

    inputs = tokenizer(
        instruction,
        return_tensors='pt',
        truncation=True,
        max_length=2048
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=1,
        stopping_criteria=stopping_criteria,
        use_cache=True
    )

    generated_answer = tokenizer.batch_decode(outputs)[0].split("<|im_start|>assistant\n")[1].replace("<|endoftext|>", "")

    df_test.at[index, "generated_answer"] = generated_answer
    
    print(f'==================== Generate output: ====================\n{generated_answer}')
    print(f"==================== Ground truth:====================\n{values['answer']}\n")

In [ ]:
df_test

In [ ]:
df_test["generated_answer"].value_counts()

In [ ]:
df_test = df_test[["answer", "generated_answer"]]
df_test["generated_answer"] = df_test["generated_answer"].apply(lambda x:x.replace("<|im_end|>", ""))
df_test

In [ ]:
df_test["answer"] = df_test["answer"].apply(lambda x: 1 if x == "A" else 0)
df_test["generated_answer"] = df_test["generated_answer"].apply(lambda x: 1 if x == "A" else 0)
df_test

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("Accuracy:", round(accuracy_score(df_test["answer"], df_test["generated_answer"])*100, 2))
print("Precision:", round(precision_score(df_test["answer"], df_test["generated_answer"])*100, 2))
print("Recall:", round(recall_score(df_test["answer"], df_test["generated_answer"])*100, 2))
print("F1 score:", round(f1_score(df_test["answer"], df_test["generated_answer"])*100, 2))